In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

### Loading files

In [ ]:
arc = pd.read_csv("../data/raw/data_arc.csv")
bulk_additions = pd.read_csv("../data/raw/data_bulk.csv")
bulk_additions_time = pd.read_csv("../data/raw/data_bulk_time.csv")
stirring_gas = pd.read_csv("../data/raw/data_gas.csv")
temperature = pd.read_csv("../data/raw/data_temp.csv")
temperature_full = pd.read_csv("../data/raw/data_temp_FULL_with_test.csv")
wire_additions = pd.read_csv("../data/raw/data_wire.csv")
wire_additions_time = pd.read_csv("../data/raw/data_wire_time.csv")

datasets = {
    "arc": arc,
    "bulk_additions": bulk_additions,
    "bulk_additions_time": bulk_additions_time,
    "stirring_gas": stirring_gas,
    "temperature": temperature,
    "temperature_full": temperature_full,
    "wire_additions": wire_additions,
    "wire_additions_time": wire_additions_time,
}

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  {name}  —  {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"{'='*60}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    if df.isnull().sum().sum() == 0:
        print("No nulls.")
    print(f"\nSample rows:")
    display(df.head(3))

### Data types

Pandas inferred all timestamp columns as `object` (string) instead of `datetime64`. 
This affects the following files:

- `data_arc`: columns `Heating start` and `Heating end`
- `data_bulk_time`: columns `Bulk 1` through `Bulk 15`
- `data_wire_time`: columns `Wire 1` through `Wire 9`
- `data_temp` and `data_temp_full`: column `time`

All remaining columns were correctly inferred — numeric columns loaded as `int64` 
or `float64` as expected.

The timestamps follow a consistent format (`YYYY-MM-DD HH:MM:SS`) across all files, 
so conversion is straightforward. Passing the format explicitly to `pd.to_datetime` 
avoids the overhead of automatic format inference, which matters when iterating 
over 15 bulk and 9 wire columns.

### Converting data types

In [ ]:
arc["Heating start"] = pd.to_datetime(arc["Heating start"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
arc["Heating end"] = pd.to_datetime(arc["Heating end"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

bulk_additions_time_cols = [f"Bulk {i}" for i in range(1, 16)]
for col in bulk_additions_time_cols:
    bulk_additions_time[col] = pd.to_datetime(bulk_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

temperature["time"] = pd.to_datetime(temperature["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
temperature_full["time"] = pd.to_datetime(temperature_full["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

wire_additions_time_cols = [f"Wire {i}" for i in range(1, 10)]
for col in wire_additions_time_cols:
    wire_additions_time[col] = pd.to_datetime(wire_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")


### Melt coverage across files

In [ ]:
print("Unique melts per file:")
for name, df in datasets.items():
    print(f"  {name:15s}: {df['key'].nunique()} melts")

all_keys = set(temperature_full["key"].unique())
for name, df in datasets.items():
    if name in ("temperature", "temperature_full"):
        continue
    missing = all_keys - set(df["key"].unique())
    extra = set(df["key"].unique()) - all_keys
    if missing:
        print(f"\n  {name}: {len(missing)} melts in temperature_full but NOT in {name}")
    if extra:
        print(f"\n  {name}: {len(extra)} melts in {name} but NOT in temperature_full")

if not any(
    (all_keys - set(df["key"].unique())) or (set(df["key"].unique()) - all_keys)
    for name, df in datasets.items()
    if name not in ("temperature", "temperature_full")
):
    print("\nAll files share the same set of melt keys.")

### Melt consistency across files

Not all files share the same set of melt keys. Some gaps are physically 
plausible; others are more likely registration failures.

- `arc`: 2 melts missing — arc heating is practically mandatory in LF 
  operations. The logistics in a melt shop impose waiting time between secondary 
  refining and casting, causing temperature drop that almost always requires 
  compensation. Ladle lids can reduce heat loss but rarely eliminate the need 
  for heating entirely. These 2 cases are most likely registration failures 
  rather than genuine absence of arc heating.

- `bulk_additions`: 87 melts missing — plausible for simple steel grades 
  where primary refining already delivers composition within specification. 
  The low count relative to total melts (2.7%) suggests these are genuine 
  exceptions rather than a data quality issue. These melts will be assigned 
  zero bulk addition features during feature engineering.

- `wire_additions`: 135 melts missing — wire treatment (specially CaSi for 
  inclusion modification) is a very common practice in 
  LF operations, particularly for continuous casting routes where inclusion 
  control is critical. 135 missing melts is high enough to warrant 
  investigation: if these keys are concentrated in a specific sequential 
  range, it may indicate an instrumentation failure during that period rather 
  than genuine absence of wire treatment.

- `stirring_gas`: 25 melts present in gas data but absent from temperature 
  records — bottom stirring is a mandatory operation. The absence of 
  temperature measurements for these melts suggests incomplete process cycles 
  or data collection failures. These 25 melts will be excluded from modeling 
  since there is no target variable to predict.

### Verifying temporal sequence of heats

In [ ]:
arc_timing = arc.groupby("key").agg(
    first_heat=("Heating start", "min"),
    last_heat=("Heating end", "max")
).reset_index().sort_values("first_heat")

print(arc_timing.head(20).to_string(index=False))

### Investigating the distribution of heats that miss wire information

In [ ]:
missing_wire_keys = sorted(all_keys - set(wire_additions["key"].unique()))

arc_missing = arc[arc["key"].isin(missing_wire_keys)][["key", "Heating start"]].sort_values("Heating start")

arc_missing["time_gap"] = arc_missing["Heating start"].diff()
block_threshold = pd.Timedelta(hours=2)
arc_missing["new_block"] = arc_missing["time_gap"] > block_threshold
arc_missing["block_id"] = arc_missing["new_block"].cumsum()

blocks = arc_missing.groupby("block_id").agg(
    n_heats=("key", "count"),
    start=("Heating start", "min"),
    end=("Heating start", "max")
).reset_index(drop=True)

print(f"Missing wire keys: {len(missing_wire_keys)}")
print(f"Number of temporal blocks: {len(blocks)}")
print(f"\nBlocks with 3 or more consecutive heats:")
print(blocks[blocks["n_heats"] >= 3].to_string(index=False))

### Wire addition gap investigation

To assess whether missing wire keys represent instrumentation failures or 
genuine process variation, the temporal sequence of heats was first verified 
using arc heating timestamps. The data confirms that `key` is assigned in 
strict chronological order — consecutive keys correspond to consecutive heats 
in time, with inter-heat gaps of approximately 5–10 minutes, consistent with 
ladle transfer time between primary refining and the LF station.

With chronological ordering confirmed, missing wire keys were grouped into 
temporal blocks — sequences of consecutive heats separated by less than 2 
hours. The analysis revealed 43 distinct blocks, of which 35 contain 3 or 
more consecutive heats without wire registration.

The largest blocks span 28, 24, 23 and 22 consecutive heats, covering periods 
of up to 4 hours of uninterrupted production. This pattern repeats consistently 
throughout the entire dataset (May to August 2019), across different days and 
hours, with no concentration in a specific period.

This rules out isolated instrumentation failure as the primary explanation. 
A more plausible interpretation is that this plant produces a mix of steel 
grades, a portion of which does not require wire treatment by specification. 
Simple structural steels, for example, often do not require inclusion 
modification with CaSi wire. Production scheduling typically groups similar 
grades in sequential campaigns, which would naturally produce the block 
structure observed here.

Implication for modeling: the 135 melts without wire records will be assigned 
zero wire addition features during feature engineering. Unlike the isolated 
gaps discussed earlier, these zeros may genuinely reflect absence of wire 
treatment rather than missing data — which means the model should be able to 
learn from them meaningfully. However, without grade information in the dataset, 
this cannot be confirmed, and the uncertainty should be kept in mind when 
interpreting wire-related features in the explainability notebook.

### Verifying temperature distribution and missingness

In [ ]:
print(f"temperature rows:      {len(temperature)}")
print(f"temperature_full rows: {len(temperature_full)}")

missing_in_temp = temperature["Temperature"].isna().sum()
print(f"\nMissing in temperature:      {missing_in_temp}")
print(f"Missing in temperature_full: {temperature_full['Temperature'].isna().sum()}")

extra_rows = len(temperature_full) - len(temperature)
print(f"\nExtra rows in temperature_full vs temperature: {extra_rows}")

In [ ]:
test_mask = temperature["Temperature"].isna()
print(f"Test rows (NaN in temperature): {test_mask.sum()}")
print(f"These rows in temperature_full — all have values: "
      f"{temperature_full.loc[test_mask.values, 'Temperature'].notna().all()}")

print(f"\nSample of test rows with ground truth:")
sample = temperature[test_mask].copy()
sample["true_temperature"] = temperature_full.loc[test_mask.values, "Temperature"].values
print(sample.head(10).to_string(index=False))

### Train/test structure

The dataset provides two temperature files with identical structure and row count 
(15907 rows each), but serving distinct roles:

- `data_temp.csv` contains 2901 missing temperature values — these are the rows 
  to predict, forming the test set.
- `data_temp_FULL_with_test.csv` contains all temperatures filled in, including 
  the 2901 test rows, and serves as the ground truth for final evaluation.

The test rows are not concentrated in specific melts or time periods — they are 
distributed across the dataset, which means the model will need to generalize 
across different operating conditions rather than memorize a specific subset.

Since the ground truth is available, model performance on the test set can be 
evaluated with real metrics after training. The ground truth file will not be 
used during training or validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

temperature_full["Temperature"].dropna().hist(bins=50, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Measured Temperatures")

measurements_per_melt = temperature_full.groupby("key")["Temperature"].agg(
    total="count", missing=lambda x: x.isna().sum(), measured=lambda x: x.notna().sum()
)
measurements_per_melt["total"].hist(bins=30, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Number of Measurements")
axes[1].set_ylabel("Number of Melts")
axes[1].set_title("Measurements per Melt")

plt.tight_layout()
plt.show()

In [ ]:
print("Measurements per melt — summary statistics:")
display(measurements_per_melt.describe())

print(f"\nMelts with at least one missing temperature: "
      f"{(measurements_per_melt['missing'] > 0).sum()} / {len(measurements_per_melt)}")

In [ ]:
print("Temperatures below 1500°C:")
print(temperature_full[temperature_full["Temperature"] < 1500].shape[0])

print("\nTemperatures above 1680°C:")
print(temperature_full[temperature_full["Temperature"] > 1680].shape[0])

In [ ]:
cold = temperature_full[temperature_full["Temperature"] < 1500][["key", "time", "Temperature"]]
print(cold.to_string(index=False))

In [ ]:
for key in cold["key"].unique():
    melt_temps = temperature_full[temperature_full["key"] == key][["time", "Temperature"]].sort_values("time")
    print(f"\nMelt {key}:")
    print(melt_temps.to_string(index=False))

### Extreme temperature values

Eight temperature measurements fall below 1500°C, ranging from 1189°C to 1383°C. 
Since the melting point of carbon steel is approximately 1480–1520°C depending on 
composition, these values represent physically impossible readings for liquid steel 
in active LF operation.

Examining each case in context reveals a consistent pattern:

- Six measurements (melts 867, 1214, 1619, 2052, 2561 and 3034) show an abrupt 
  drop to implausible temperatures followed by an immediate return to normal values 
  within seconds to minutes. In melt 1214, a reading of 1208°C is followed 45 
  seconds later by 1608°C — a 400°C jump that is impossible for 
  a ladle of liquid steel.

- Melt 1818 shows two consecutive readings of 1383°C embedded between values above 
  1650°C, with less than 2 minutes separating all three points.

- Five of the eight anomalous readings occur as the first measurement of their 
  respective melts, suggesting the pyrometer had not yet stabilized or was not 
  properly immersed in the bath at the time of reading.

These are instrument errors, not real process events. They will be removed before 
feature engineering. Retaining them would corrupt the `prev_temp` feature for all 
subsequent measurements within the affected melts — a single bad reading would 
propagate incorrect thermal context through the entire melt's feature vector.

The 28 measurements above 1680°C were also reviewed. Unlike the cold readings, 
high temperatures in this range are physically plausible — intentional superheating 
occurs when a heat is expected to wait longer than usual before casting, for example 
when the tundish is still occupied. 
The operator heats above the normal target to compensate for the additional thermal 
loss during the extended waiting period. Peak readings during these heating cycles 
can reach 1680°C or above transiently.

## Temporal structure of a heat

In [ ]:
candidates = measurements_per_melt[
    (measurements_per_melt["missing"] == 0) & (measurements_per_melt["total"] >= 4)
]
sample_key = candidates.index[0]
print(f"Selected heat: {sample_key}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

melt_arc = arc[arc["key"] == sample_key]
melt_temp = temperature_full[temperature_full["key"] == sample_key].sort_values("time")
melt_bulk_time = bulk_additions_time[bulk_additions_time["key"] == sample_key]
melt_wire_time = wire_additions_time[wire_additions_time["key"] == sample_key]

# temperature measurements
ax.plot(melt_temp["time"], melt_temp["Temperature"], 
        color="red", marker="o", linewidth=2, label="Temperature")

# arc heating intervals
for _, row in melt_arc.iterrows():
    ax.axvspan(row["Heating start"], row["Heating end"], 
               alpha=0.15, color="orange", label="Arc heating" 
               if _ == melt_arc.index[0] else "")

# bulk additions
bulk_cols = [c for c in melt_bulk_time.columns if c.startswith("Bulk")]
bulk_plotted = False
for col in bulk_cols:
    val = melt_bulk_time[col].values[0]
    if pd.notna(val):
        label = "Bulk addition" if not bulk_plotted else ""
        ax.axvline(pd.Timestamp(val), color="blue", 
                   linestyle="--", alpha=0.6, label=label)
        bulk_plotted = True

# wire additions
wire_cols = [c for c in melt_wire_time.columns if c.startswith("Wire")]
wire_plotted = False
for col in wire_cols:
    val = melt_wire_time[col].values[0]
    if pd.notna(val):
        label = "Wire addition" if not wire_plotted else ""
        ax.axvline(pd.Timestamp(val), color="green", 
                   linestyle=":", alpha=0.6, label=label)
        wire_plotted = True

ax.set_xlabel("Time")
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"Full Timeline of Heat {sample_key}")
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.legend(loc="upper left")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Heat 1 timeline

The timeline above shows the full sequence of events for Heat 1, from 11:01 to 
11:31. This plot illustrates the cyclic nature of the LF process: arc heating 
sessions alternate with measurement points and material additions throughout 
the heat.

**Arc heating** — five heating sessions are visible as orange shaded regions. 
The first three sessions (11:02–11:14) are short and closely spaced, suggesting 
an initial heating phase to bring the bath to a workable temperature range. The 
fourth session (11:18–11:24) is the longest and most powerful (1.66 MW), 
delivering the bulk of the energy input. The fifth session (11:26–11:28) is a 
short trim heat before the final measurements.

**Bulk additions** — three materials (Bulk 12, 14 and 15) were added 
simultaneously at 11:03, during the first heating session. This is a common 
practice: the operator adds alloys early while the arc is active, allowing 
stirring and heating to dissolve and homogenize the additions. A fourth bulk 
addition (Bulk 4) was made at 11:21, during the fourth heating session.

**Wire addition** — Wire 1 was injected at 11:11, between the third and fourth 
heating sessions. Wire injection is typically performed with the arc off to avoid 
arc interference with the wire feeding mechanism and to allow the injected 
material to dissolve without excessive turbulence.

**Temperature evolution** — the first measurement at 11:16 reads 1571°C, taken 
after three heating sessions and all early additions. The bath then rises steadily 
to 1604°C at 11:25 and peaks at 1618°C at 11:29, reflecting the energy delivered 
by the fourth and fifth heating sessions. The drop to 1601°C at 11:30 followed by 
a partial recovery to 1613°C within 90 seconds is unlikely to reflect a real 
thermal event — a steel ladle cannot lose and recover 17°C in that 
timeframe. This is most likely a measurement artefact.

This timeline of this heat illustrates the physical logic that will drive feature 
engineering: temperature at any measurement point is the result 
of cumulative energy input, cumulative thermal losses from additions and time, 
and the thermal state inherited from the previous measurement.

### Arc heating analysis

In [ ]:
arc_analysis = arc.copy()

arc_analysis["duration_min"] = (
    (arc_analysis["Heating end"] - arc_analysis["Heating start"]).dt.total_seconds() / 60
)
arc_analysis["duration_s"] = (
    arc_analysis["Heating end"] - arc_analysis["Heating start"]
).dt.total_seconds()

arc_analysis["energy_MJ"] = arc_analysis["Active power"] * arc_analysis["duration_s"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

arc_analysis["Active power"].hist(bins=40, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Active power (MW)")
axes[0].set_ylabel("Count")
axes[0].set_title("Active power distribution")

arc_analysis["duration_min"].hist(bins=40, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Duration (min)")
axes[1].set_ylabel("Count")
axes[1].set_title("Heating duration distribution")

arc_analysis["energy_MJ"].hist(bins=40, ax=axes[2], edgecolor="black", alpha=0.7)
axes[2].set_xlabel("Energy (MJ)")
axes[2].set_ylabel("Count")
axes[2].set_title("Energy per heating session")

plt.tight_layout()
plt.show()

In [ ]:
print(arc_analysis[["Active power", "duration_min", "energy_MJ"]].describe().round(2))

The three plots above characterize the electric arc heating sessions across all 
melts. Understanding the distribution of power, duration and energy is important 
because arc heating is the primary mechanism of temperature increase in the LF — 
and these distributions directly motivate the feature engineering choices.

**Active power** — the distribution peaks at 0.4–0.5 MW with a mean of 0.67 MW 
and a long tail extending to 3.73 MW. Most sessions operate at moderate power 
levels, with high-power sessions being relatively uncommon. High power is 
typically reserved for situations requiring aggressive heating — compensating for 
a cold heat, recovering from a large alloy addition, or building in superheating 
margin before a long wait at the caster.

**Heating duration** — sessions are mostly short, with a median of 2.5 minutes 
and a peak at 2–3 minutes. The tail extending to 15 minutes represents longer 
heating sessions, likely the initial heating phase for heats that arrive cold 
from primary refining or after extended ladle waiting times. The predominance of 
short sessions reflects the common operational pattern in LF: heat briefly, 
measure, adjust, repeat.

**Energy per session** — this is the most physically meaningful of the three 
distributions. The strong right skew shows that most sessions deliver modest 
energy (50% below 81 MJ, 75% below 182 MJ), while a small number of sessions 
deliver substantially more — up to 3384 MJ. This reflects the same pattern 
seen in heat 1 timeline: several short low-energy sessions for 
fine adjustment, punctuated by occasional long high-power session.

Plotting power and duration separately would obscure this — a short high-power 
session and a long low-power session can deliver identical energy to the bath. 
This motivates the use of cumulative energy (MW × seconds = MJ) as the primary 
arc heating feature in Notebook 02, rather than treating power and duration as 
independent variables.

In [ ]:
sessions_per_melt = arc_analysis.groupby("key").size()
energy_per_melt = arc_analysis.groupby("key")["energy_MJ"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sessions_per_melt.hist(bins=20, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Number of Heating Sessions")
axes[0].set_ylabel("Number of Heats")
axes[0].set_title("Heating Sessions per Heat")

energy_per_melt.hist(bins=40, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Total Energy (MJ)")
axes[1].set_ylabel("Number of Heats")
axes[1].set_title("Total Energy Input per Heat")

plt.tight_layout()
plt.show()

print("Heating sessions per heat:")
print(sessions_per_melt.describe())
print(f"\nTotal energy per heat (MJ):")
print(energy_per_melt.describe())

The two plots above characterized individual heating sessions. They 
aggregate to the heat level, showing how many sessions each heat requires and 
how much total energy is delivered across the entire LF treatment.

**Heating sessions per heat** — the distribution peaks at 4–5 sessions per heat, 
with a median of 4 and a mean of 4.6. This reflects the iterative LF cycle: most 
heats require four to five rounds of heating interspersed with measurements and 
additions before reaching the target temperature. Heats with 1–2 sessions are 
likely simple cases that arrived close to target temperature. Heats with 10 or 
more sessions — extending to a maximum of 16 — represent complex cases requiring 
extended treatment, possibly due to cold arrival temperature, large alloy addition 
loads, or tight chemistry specifications requiring multiple correction cycles.

**Total energy per heat** — the distribution is right-skewed with a median of 
606 MJ and a mean of 713 MJ. The gap between median and mean reflects the 
influence of high-energy outliers — a small number of heats consume up to 
8624 MJ, more than 14 times the median. These extreme cases are consistent 
with the high-session heats seen in the left plot: more sessions means more 
cumulative energy delivered.

The wide range of total energy inputs — from 11 MJ to 8624 MJ — captures 
the full diversity of operating conditions in this dataset. This variability 
is what makes the prediction problem non-trivial and what gives the cumulative 
energy feature its predictive power: a heat that has received 2000 MJ by the 
time of a measurement is in a fundamentally different thermal state than one 
that has received 200 MJ, and the model needs to learn this distinction.

### Bulk and wire addition analysis

In [ ]:
bulk_cols = [c for c in bulk_additions.columns if c.startswith("Bulk")]
wire_cols = [c for c in wire_additions.columns if c.startswith("Wire")]

bulk_usage = bulk_additions[bulk_cols].apply(lambda col: (col > 0).sum())
wire_usage = wire_additions[wire_cols].apply(lambda col: (col > 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bulk_usage.plot(kind="bar", ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Bulk Material")
axes[0].set_ylabel("Number of Heats with Addition > 0")
axes[0].set_title("Bulk Material Usage Frequency")
axes[0].tick_params(axis="x", rotation=45)

wire_usage.plot(kind="bar", ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Wire Material")
axes[1].set_ylabel("Number of Heats with Addition > 0")
axes[1].set_title("Wire Material Usage Frequency")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
bulk_totals = bulk_additions[bulk_cols].sum(axis=1)
wire_totals = wire_additions[wire_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bulk_totals.hist(bins=40, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Total Bulk Mass (kg)")
axes[0].set_ylabel("Number of Heats")
axes[0].set_title("Total Bulk Additions per Heat")

wire_totals.hist(bins=40, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Total Wire Mass (kg)")
axes[1].set_ylabel("Number of Heats")
axes[1].set_title("Total Wire Additions per Heat")

plt.tight_layout()
plt.show()

print("Total bulk mass per heat (kg):")
print(bulk_totals.describe())
print(f"\nTotal wire mass per heat (kg):")
print(wire_totals.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bulk_means = bulk_additions[bulk_cols].mean()
bulk_means[bulk_means > 0].plot(kind="bar", ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Bulk Material")
axes[0].set_ylabel("Mean Addition (kg)")
axes[0].set_title("Mean Bulk Addition (when used)")
axes[0].tick_params(axis="x", rotation=45)

wire_means = wire_additions[wire_cols].mean()
wire_means[wire_means > 0].plot(kind="bar", ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Wire Material")
axes[1].set_ylabel("Mean Addition (kg)")
axes[1].set_title("Mean Wire Addition (when used)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

The six plots above characterize the alloy addition behavior across heats, covering which 
materials are used, how frequently, and in what quantities.

**Usage frequency** — bulk materials show a clear split between high-frequency 
and low-frequency materials. Bulk 14, 12 and 15 appear in the majority of heats, 
suggesting these are common alloys used across most steel grades — likely standard 
ferroalloys such as FeSi or FeSiMn. Bulks 8, 9, 11 and 13 appear in fewer than 
50 heats each, indicating grade-specific materials used only in particular 
heats. Without column decoding this cannot be confirmed, but the frequency 
pattern alone is informative.

Wire 1 is nearly universal, appearing in approximately 3050 heats — consistent 
with CaSi wire, which is standard practice in LF operations for inclusion 
modification ahead of continuous casting. Wire 2 appears in roughly 1050 heats, 
possibly Al wire for final deoxidation during secondary refining. The remaining 
wires are rare and likely correspond to specific alloys for more elaborated steel 
grades.

**Total additions per melt** — bulk additions range widely with a median of 
591 kg and a maximum above 3000 kg, reflecting the variability in chemistry 
correction requirements across different steel grades. Wire additions are 
considerably smaller in mass, with a median of 114 kg, and show a tighter 
distribution consistent with the precision-controlled nature of wire injection.

**Implication for modeling** — both bulk and wire materials are added at ambient 
temperature and therefore represent thermal sinks: the melt must supply the 
enthalpy required to heat these cold solids to bath temperature. Bulk additions 
dominate this effect due to their larger mass. Materials with near-zero usage 
across the dataset will contribute columns of mostly zeros to the feature matrix 
— the model will naturally learn to down-weight them, but their presence is worth 
keeping in mind when interpreting feature importance.

### Gas usage analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
stirring_gas["Gas 1"].hist(bins=40, ax=ax, edgecolor="black", alpha=0.7)
ax.set_xlabel("Total Gas Used (Nm3)")
ax.set_ylabel("Number of Heats")
ax.set_title("Gas Usage Distribution")
plt.tight_layout()
plt.show()

print("Gas usage per heat:")
print(stirring_gas["Gas 1"].describe())

Bottom ladle stirring via gas purging in LF operations are importe to 
homogenize temperature and chemical composition across the bath, eliminating 
concentration gradients, driving refining kinetics — in desulfurization, 
for example, purging promotes the transport of sulfur to the steel/slag interface 
where the reaction occurs.

The distribution peaks between 7–10 Nm³ with a median of 9.8 Nm³, consistent 
with a standardized purging protocol defined by the technological instruction. 
Most heats receive a similar gas volume, with deviations upward for heats 
requiring more intensive homogenization — deeper desulfurization targets, larger 
alloy addition loads, or tighter chemistry specifications.

The tail extending to 78 Nm³ represents heats with unusually intensive stirring, 
roughly eight times the median. These are likely cases of prolonged treatment for 
demanding metallurgical objectives. The small group of heats with values near 
zero is noted but not investigated further here — this will be revisited at the 
start of Notebook 02 alongside other data cleaning decisions.

One important limitation for feature engineering: gas usage is recorded as a 
single heat-level value with no timestamp, unlike arc heating and alloy additions 
which carry event-level timestamps. This means it is impossible to know how much 
gas had been used at the time of any specific temperature measurement within a 
heat. In the feature matrix, gas will enter as the total heat-level value — an 
approximation that is coarser than the other cumulative features but still 
captures the overall stirring intensity of the heat.